## MCP 架構的三個關鍵角色

### 1. MCP Host (主機)
- **角色**：協調和管理一個或多個 MCP 客戶端的 AI 應用程式
- **比喻**：就像是一個總經理，負責管理和協調所有的工作
- **例子**：VS Code、Claude Desktop

### 2. MCP Client (客戶端)
- **角色**：維持與 MCP 伺服器的連接，並為 MCP 主機獲取上下文資訊
- **比喻**：就像是一個中間人或傳令員，負責傳遞訊息
- **功能**：它是主機和伺服器之間的橋樑

### 3. MCP Server (伺服器)
- **角色**：向 MCP 客戶端提供上下文資訊的程式
- **比喻**：就像是一個專門的服務提供商，例如天氣服務、檔案管理服務
- **例子**：Sentry MCP 伺服器、天氣查詢伺服器

### 實際運作範例
當 VS Code (MCP Host) 連接到 Sentry MCP 伺服器時：
1. VS Code 會建立一個 MCP Client 物件
2. 這個 Client 維持與 Sentry 伺服器的連接
3. 當需要資訊時，Client 從伺服器獲取資料並傳給 VS Code

```
VS Code (Host) <---> MCP Client <---> Sentry Server
     總經理            傳令員           服務提供商
```

## 建立 MCP 伺服器

### 使用 FastMCP 函式庫

[fastmcp](https://gofastmcp.com/getting-started/welcome) 是一個 Python 函式庫，讓我們可以輕鬆建立 MCP 伺服器。

以下程式碼（也可在 `./05_src/static_mcp/server.py` 找到）展示如何建立一個簡單的 MCP 伺服器：

```python
from fastmcp import FastMCP

# 建立一個名為 "My MCP Server" 的 MCP 伺服器
mcp = FastMCP("My MCP Server")

# 使用 @mcp.tool 裝飾器來定義一個工具
@mcp.tool
def greet(name: str) -> str:
    """這是一個簡單的問候工具"""
    return f"Hello, {name}!"

# 啟動伺服器
if __name__ == "__main__":
    mcp.run(
        transport="http",      # 使用 HTTP 協議
        host="localhost",      # 在本地主機上運行
        port=3000,             # 使用 3000 埠
    )
```

### 程式碼解析

1. **`FastMCP("My MCP Server")`**：建立一個 MCP 伺服器實例
2. **`@mcp.tool`**：這是一個裝飾器，告訴系統這個函數是一個可以被 AI 使用的工具
3. **`greet` 函數**：接收一個名字，回傳問候語
4. **`mcp.run()`**：啟動伺服器，監聽指定的埠（3000）

### 為什麼需要工具？
AI 模型本身不能執行程式或查詢資料，但透過 MCP 工具，它可以：
- 呼叫你定義的函數
- 獲取即時資料
- 執行特定操作

## 反向代理 (Reverse Proxy)

### 為什麼需要反向代理？

#### 問題
OpenAI SDK **要求**與 MCP 伺服器建立 **HTTPS 連接**（而不是 HTTP）。

- **HTTP**：不安全的連接（像明信片，任何人都能看）
- **HTTPS**：加密的安全連接（像密封的信封，只有收件人能看）

要在本地運行 HTTPS 伺服器，我們需要一個由**受信任機構**（如 Let's Encrypt）提供的 TLS 證書。這通常很複雜且費時。

#### 解決方案：ngrok

[ngrok](https://ngrok.com/) 是一個服務，它可以：
1. 提供一個公開的 HTTPS URL
2. 將所有請求轉發到你的本地伺服器（localhost:3000）
3. 自動處理 TLS 證書問題

### 運作原理

```
OpenAI API --HTTPS--> ngrok URL --HTTP--> localhost:3000 (你的 MCP 伺服器)
   安全連接              轉發              本地連接
```

### 比喻
ngrok 就像是一個郵局轉信服務：
- 你的本地伺服器是你家（沒有公開地址）
- ngrok 提供一個公開的地址
- 所有寄到這個公開地址的信件，都會被轉發到你家

## 實作 1：靜態 MCP 伺服器

### 啟動步驟

在終端機中，將工作目錄切換到 `./05_src/`，然後執行以下命令：

#### 第一步：啟動 MCP 伺服器
```bash
python -m static_mcp.server
```
這會在 localhost:3000 啟動你的 MCP 伺服器。

#### 第二步：設定反向代理
```bash
ngrok http 3000
```
這會建立一個公開的 HTTPS URL，指向你的本地 3000 埠。

### 重要說明

- **MCP_URL**：ngrok 執行後會顯示一個 URL（例如：`https://abc123.ngrok.io`）
- 所有發送到這個 URL 的請求都會被轉發到 `localhost:3000`
- 你需要將這個 URL 設定到環境變數 `MCP_URL` 中

### 使用兩個終端機
1. **終端機 1**：運行 `python -m static_mcp.server`（保持運行）
2. **終端機 2**：運行 `ngrok http 3000`（保持運行）
3. **此筆記本**：執行下面的程式碼與伺服器互動

In [ ]:
# 載入環境變數
# 這些 .env 和 .secrets 檔案包含你的 API 金鑰和設定
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets

In [ ]:
import os
from openai import OpenAI

# 建立 OpenAI 客戶端
client = OpenAI()

# 從環境變數獲取 MCP 伺服器的 URL
mcp_url = os.getenv("MCP_URL")

print(f'使用 MCP URL: {mcp_url}')

# 定義工具配置
tools = [
    {
        "type": "mcp",                    # 工具類型是 MCP
        "server_label": "greeting_service",  # 給這個服務一個標籤
        "server_description": "一個提供個人化問候的服務。",  # 描述這個服務的功能
        "server_url": mcp_url,            # MCP 伺服器的 URL
        "require_approval": "never",      # 不需要使用者批准就可以使用工具
    },
]

# 向 OpenAI API 發送請求
resp = client.responses.create(
    model="gpt-5",                        # 使用 GPT-5 模型
    tools=tools,                          # 提供可用的工具
    instructions="使用問候服務來回答問題。",  # 給 AI 的指示
    input="你好，我是 Alice。",             # 使用者的輸入
)

# 顯示 AI 的回應
print(resp.output_text)

### 程式碼運作流程

1. **使用者輸入**：「你好，我是 Alice。」
2. **GPT-5 分析**：判斷需要使用問候服務
3. **呼叫 MCP 工具**：向 MCP 伺服器發送請求，呼叫 `greet("Alice")` 函數
4. **MCP 伺服器回應**：回傳 "Hello, Alice!"
5. **GPT-5 整合**：將結果整合到自然語言回應中
6. **輸出**：顯示友善的回應給使用者

### 為什麼這很有用？
- AI 可以使用你定義的自訂功能
- 你可以控制 AI 能做什麼
- 可以整合任何你想要的服務或資料

## 實作 2：靜態天氣服務

### 切換到天氣服務

這是一個更完整的範例，展示如何提供天氣資訊。

#### 切換步驟
1. 在運行 `python -m static_mcp.server` 的終端機中按 **CTRL+C** 停止伺服器
2. 啟動天氣服務：
   ```bash
   python -m static_weather_mcp.server
   ```
3. ngrok 不需要重新啟動（保持運行）

### 為什麼稱為「靜態」？
因為這個範例回傳的是固定的天氣資料，而不是從真實的氣象 API 獲取即時資料。這樣做是為了：
- 簡化學習過程
- 避免需要額外的 API 金鑰
- 專注於理解 MCP 的運作原理

在真實應用中，你會連接到實際的天氣 API（如 OpenWeatherMap）來獲取即時資料。

In [ ]:
from openai import OpenAI
client = OpenAI()

# 使用相同的 MCP URL（ngrok 還在運行）
mcp_url = os.getenv("MCP_URL")

print(f'使用 MCP URL: {mcp_url}')

# 定義天氣服務工具
tools = [
    {
        "type": "mcp",
        "server_label": "weather_service",     # 服務標籤改為天氣服務
        "server_description": "一個提供當前天氣狀況的服務。",  # 描述改為天氣相關
        "server_url": mcp_url,
        "require_approval": "never",
    },
]

# 詢問天氣
resp = client.responses.create(
    model="gpt-5",
    tools=tools,
    instructions="使用天氣服務來回答問題。",
    input="目前的天氣如何？",
)

print(resp.output_text)

### 預期結果

AI 會使用天氣服務工具來回答問題，可能會回覆類似：
- "目前天氣晴朗，溫度 25°C，濕度 60%"

### 學習要點

1. **同一個架構，不同的服務**：只需要改變伺服器端的程式碼，客戶端的呼叫方式幾乎相同
2. **靈活性**：你可以為 AI 添加任何你想要的功能
3. **標準化**：所有的服務都使用相同的 MCP 協議，易於維護和擴展

## 補充資料：進階概念

### 1. MCP 與 Function Calling 的關係

MCP 是建立在函數呼叫（Function Calling）概念之上的：
- **Function Calling**：AI 能夠決定何時以及如何呼叫你定義的函數
- **MCP**：提供一個標準化的方式來組織和提供這些函數

### 2. MCP 的實際應用場景

#### 企業應用
- 連接到公司的內部資料庫
- 整合 CRM 系統（客戶關係管理）
- 查詢庫存系統

#### 開發工具
- 連接到 GitHub、GitLab
- 整合錯誤追蹤系統（如 Sentry）
- 自動化部署流程

#### 個人助理
- 行事曆管理
- 郵件整理
- 檔案搜尋

### 3. 安全性考量

#### require_approval 參數
```python
"require_approval": "never"  # 不需要批准
"require_approval": "always" # 每次都需要使用者批准
```

#### 何時需要使用者批准？
- 刪除資料
- 發送郵件
- 進行金融交易
- 修改重要檔案

### 4. 開發提示

#### 除錯技巧
1. **檢查伺服器是否運行**：在瀏覽器訪問 `http://localhost:3000`
2. **檢查 ngrok**：確認 ngrok 顯示的 URL 正確
3. **查看日誌**：MCP 伺服器會在終端機顯示請求日誌

#### 常見錯誤
1. **連接失敗**：檢查防火牆設定
2. **HTTPS 錯誤**：確認使用的是 ngrok 提供的 HTTPS URL
3. **工具未被呼叫**：檢查 instructions 是否清楚指示 AI 使用工具

### 5. 擴展學習資源

- [MCP 官方文件](https://modelcontextprotocol.io/)
- [FastMCP 指南](https://gofastmcp.com/)
- [ngrok 文件](https://ngrok.com/docs)
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)

## 總結

### 核心概念回顧

1. **MCP 是什麼**：連接 AI 與外部工具/資料的標準協議
2. **三個角色**：Host（主機）、Client（客戶端）、Server（伺服器）
3. **為何需要 ngrok**：提供 HTTPS 連接，解決 TLS 證書問題
4. **如何建立工具**：使用 FastMCP 和 `@mcp.tool` 裝飾器

### 實作步驟

1. 建立 MCP 伺服器（定義工具）
2. 啟動伺服器（localhost:3000）
3. 使用 ngrok 建立公開 HTTPS URL
4. 在 OpenAI API 中配置工具
5. AI 可以使用你的自訂工具！

### 下一步

嘗試建立你自己的 MCP 伺服器：
- 連接到真實的 API（天氣、新聞、股票）
- 查詢本地資料庫
- 整合你最常用的工具